In [1]:
import torch
from PIL import Image
import cv2, os, subprocess
from google.colab import drive
from tqdm import tqdm

# Check for GPU availability
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

# Clone the Real-ESRGAN repository
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Install PyTorch (Colab usually has compatible versions pre-installed)
!pip install -q torch torchvision torchaudio

# Install required dependencies
!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt

# Setup the Real-ESRGAN package
!python setup.py develop

# Mount Google Drive if needed
mount_drive = False  # @param {type:"boolean"}

if mount_drive:
    drive.mount('/content/gdrive/')


/content/Real-ESRGAN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 20.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 27.7 MB/s eta 0:00:00
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ***********************************************************

In [2]:
import os
import sys

# basicsr library ka installation path directly set karna
# Hum jante hain ki yeh /usr/local/lib/python3.12/dist-packages mein install hua hai
basicsr_install_path = None
for p in sys.path:
    if "dist-packages" in p and os.path.exists(os.path.join(p, "basicsr")):
        basicsr_install_path = os.path.join(p, "basicsr")
        break

if basicsr_install_path is None:
    # Fallback if the path isn't found in sys.path (unlikely after successful pip install)
    # Yeh path standard Google Colab environment ke liye hai
    basicsr_install_path = "/usr/local/lib/python3.12/dist-packages/basicsr"

deg_file = os.path.join(basicsr_install_path, 'data', 'degradations.py')

# File ko open kar ke galat import ko theek karna
if os.path.exists(deg_file):
    with open(deg_file, 'r') as f:
        content = f.read()

    # Sirf tab replace karein jab purana wala import string maujood ho
    old_import = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
    new_import = 'from torchvision.transforms.functional import rgb_to_grayscale'

    if old_import in content:
        content = content.replace(old_import, new_import)

        # Theek kiya hua code wapis save karna
        with open(deg_file, 'w') as f:
            f.write(content)
        print("✅ basicsr degradations.py file updated successfully!")
    else:
        print("ℹ️ basicsr degradations.py file mein pehle se hi sahi import hai ya badlav ki zaroorat nahi hai.")
else:
    print(f"❌ Error: degradations.py file '{deg_file}' nahi mila.")

# Ab basicsr ko import karne ki koshish karein taaki verify ho sake ki fix kaam kar gaya.
try:
    import basicsr
    print("✅ basicsr successfully imported after fix.")
    print("Ab aap apna upscaling wala code dobara run kar sakte hain.")
except ModuleNotFoundError as e:
    print(f"❌ basicsr import abhi bhi fail ho raha hai: {e}")

✅ basicsr degradations.py file updated successfully!
✅ basicsr successfully imported after fix.
Ab aap apna upscaling wala code dobara run kar sakte hain.


In [3]:
import os, json, shutil, subprocess, uuid
from google.colab import drive

# =========================
# USER OPTIONS
# =========================
#@title Video Upscaler Settings
video_path   = "/content/input.mp4" #@param {type:"string"}
output_dir   = "/content/" #@param {type:"string"}
resolution = "8k (7680 x 4320)" #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "8k (7680 x 4320)", "2 x original", "3 x original", "4 x original"]
model        = "RealESRGAN_x4plus_anime_6B" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]
tile_size    = 512 #@param {type:"integer"}
mount_drive  = False #@param {type:"boolean"}
target_fps   = 24 #@param {type:"integer"}

# Quality/speed knobs (Inhein UI mein nahi rakha taake interface simple rahay)
cfr_crf      = 18     # CFR conversion quality (lower=better)
final_crf    = 17     # final encode quality
final_preset = "slow" # "medium" for faster, "slow" for better

assert os.path.exists(video_path), "Error: Video file does not exist!"

# =========================
# Helpers
# =========================
def run(cmd, check=True):
    """Run shell command and print it."""
    print("Running command:", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and p.returncode != 0:
        print("--- Command stdout ---")
        print(p.stdout)
        print("--- Command stderr ---")
        print(p.stderr)
        raise RuntimeError(f"Command failed: {cmd}")

def ffprobe_json(path):
    cmd = f'ffprobe -v error -print_format json -show_format -show_streams "{path}"'
    out = subprocess.check_output(cmd, shell=True).decode("utf-8")
    return json.loads(out)

def get_video_stream(info):
    for s in info.get("streams", []):
        if s.get("codec_type") == "video":
            return s
    raise RuntimeError("No video stream found")

def find_latest_mp4(folder):
    mp4s = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".mp4")]
    if not mp4s:
        return None
    mp4s.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return mp4s[0]

# =========================
# Mount Drive (optional)
# =========================
if mount_drive:
    drive.mount('/content/gdrive/')
    drive_folder = "/content/gdrive/MyDrive/Upscaled_Videos_REAL_ESRGAN"
    os.makedirs(drive_folder, exist_ok=True)

# =========================
# Read input info
# =========================
info = ffprobe_json(video_path)
v = get_video_stream(info)

in_w = int(v.get("width", 0))
in_h = int(v.get("height", 0))
duration = float(info["format"].get("duration", 0.0))

print(f"Input: {in_w}x{in_h}, duration={duration:.2f}s")

# Target resolutions
resolutions_dict = {
    "FHD (1920 x 1080)": (1920, 1080),
    "2k (2560 x 1440)":  (2560, 1440),
    "4k (3840 x 2160)":  (3840, 2160),
    "8k (7680 x 4320)":  (7680, 4320),
    "2 x original":      (in_w * 2, in_h * 2),
    "3 x original":      (in_w * 3, in_h * 3),
    "4 x original":      (in_w * 4, in_h * 4),
}
target_w, target_h = resolutions_dict[resolution]

# Scale factor for Real-ESRGAN (outscale)
initial_scale_factor = min(target_w / in_w, target_h / in_h)

# Calculate proposed upscaled dimensions and make them even.
proposed_up_w = int(in_w * initial_scale_factor)
proposed_up_h = int(in_h * initial_scale_factor)

up_w = proposed_up_w + (proposed_up_w % 2)
up_h = proposed_up_h + (proposed_up_h % 2)

scale_factor = max(up_w / in_w, up_h / in_h)

print(f"Plan: upscale to ~{up_w}x{up_h} (outscale={scale_factor:.4f}), then pad to {target_w}x{target_h}")

# =========================
# Create unique workdir
# =========================
run_id = uuid.uuid4().hex[:8]
workdir = f"/content/realesrgan_work_{run_id}"
os.makedirs(workdir, exist_ok=True)

local_input = os.path.join(workdir, "input.mp4")
shutil.copy(video_path, local_input)

# =========================
# Convert to CFR
# =========================
cfr_input = os.path.join(workdir, "input_cfr.mp4")
run(
    f'ffmpeg -y -loglevel error -i "{local_input}" '
    f'-vf "fps={target_fps}" -vsync cfr '
    f'-c:v libx264 -crf {cfr_crf} -preset veryfast -pix_fmt yuv420p '
    f'-c:a copy "{cfr_input}"'
)

# =========================
# Locate Real-ESRGAN video script
# =========================
candidate_scripts = [
    "/content/Real-ESRGAN/inference_realesrgan_video.py",
    "/content/inference_realesrgan_video.py",
]
script_path = None
for p in candidate_scripts:
    if os.path.exists(p):
        script_path = p
        break
if script_path is None:
    raise FileNotFoundError(
        "Error: inference_realesrgan_video.py nahi mil rahi.\n"
        "Aap Real-ESRGAN repo clone karke script path set karein, "
        "ya script ko /content/ me rakhein."
    )

realesr_outdir = os.path.join(workdir, "realesr_out")
os.makedirs(realesr_outdir, exist_ok=True)

cwd_before = os.getcwd()
os.chdir(workdir)

for p in ["tmp_frames", "tmp_videos", "results", "frames", "output"]:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)

# =========================
# Run Real-ESRGAN on CFR input with Tiling
# =========================
tile_arg = f"--tile {tile_size}" if tile_size > 0 else ""

cmd = (
    f'python "{script_path}" '
    f'-n {model} '
    f'-i "{cfr_input}" '
    f'-o "{realesr_outdir}" '
    f'--fps {target_fps} '
    f'--outscale {scale_factor} '
    f'{tile_arg}'
)
run(cmd)

os.chdir(cwd_before)

upscaled_video = find_latest_mp4(realesr_outdir)
if upscaled_video is None:
    raise RuntimeError("Error: Real-ESRGAN output mp4 nahi bani. Script logs check karein.")

print("Upscaled video:", upscaled_video)

# =========================
# Pad to exact target resolution
# =========================
video_name = os.path.splitext(os.path.basename(video_path))[0]
final_name = f"{video_name}_upscaled_{target_w}x{target_h}_{model}.mp4"
final_path = os.path.join(output_dir, final_name)

pad_x = max((target_w - up_w) // 2, 0)
pad_y = max((target_h - up_h) // 2, 0)

vf = (
    f"scale={up_w}:{up_h}:flags=lanczos,"
    f"pad={target_w}:{target_h}:{pad_x}:{pad_y}:black"
)

run(
    f'ffmpeg -y -loglevel error -i "{upscaled_video}" '
    f'-vf "{vf}" '
    f'-c:v libx264 -crf {final_crf} -preset {final_preset} -pix_fmt yuv420p '
    f'-c:a copy "{final_path}"'
)

print(f"Final saved: {final_path}")

# =========================
# Save to Drive (optional)
# =========================
if mount_drive:
    drive_save_path = os.path.join(drive_folder, final_name)
    run(f'mv "{final_path}" "{drive_save_path}"')
    print(f"Saved to Drive: {drive_save_path}")

# =========================
# Cleanup
# =========================
shutil.rmtree(workdir, ignore_errors=True)
print("Cleaned workdir:", workdir)

Input: 1280x720, duration=5.06s
Plan: upscale to ~7680x4320 (outscale=6.0000), then pad to 7680x4320
Running command: ffmpeg -y -loglevel error -i "/content/realesrgan_work_ca82b195/input.mp4" -vf "fps=24" -vsync cfr -c:v libx264 -crf 18 -preset veryfast -pix_fmt yuv420p -c:a copy "/content/realesrgan_work_ca82b195/input_cfr.mp4"
Running command: python "/content/Real-ESRGAN/inference_realesrgan_video.py" -n RealESRGAN_x4plus_anime_6B -i "/content/realesrgan_work_ca82b195/input_cfr.mp4" -o "/content/realesrgan_work_ca82b195/realesr_out" --fps 24 --outscale 6.0 --tile 512
Upscaled video: /content/realesrgan_work_ca82b195/realesr_out/input_cfr_out.mp4
Running command: ffmpeg -y -loglevel error -i "/content/realesrgan_work_ca82b195/realesr_out/input_cfr_out.mp4" -vf "scale=7680:4320:flags=lanczos,pad=7680:4320:0:0:black" -c:v libx264 -crf 17 -preset slow -pix_fmt yuv420p -c:a copy "/content/input_upscaled_7680x4320_RealESRGAN_x4plus_anime_6B.mp4"
Final saved: /content/input_upscaled_7680x